# `n_color_immerse()` — why nematic directors need an immersion-based colormap

`nematics3d.field.n_color_immerse()` maps a three-dimensional nematic director

$$
\mathbf n=(n_x,n_y,n_z),\qquad |\mathbf n|=1,
$$

to an sRGB color.

At first sight this may look like an ordinary visualization utility: take three numbers and turn them into three other numbers. The difficulty is that a nematic director is **not an ordinary vector**. The states $\mathbf n$ and $-\mathbf n$ represent the same physical orientation. That single identification changes the topology of the orientation space and makes a globally perfect three-dimensional color encoding impossible.

This notebook explains the design of `n_color_immerse()` in detail: what an ideal director colormap would mean, why such an ideal map cannot exist, why Nematics3D uses an immersion related to Boy's surface, how the perceptual optimization is formulated in OKLab, and why the current production map uses the selected $J_{\rm loc}\le 0.55$ solution.

## 1. What would an ideal nematic colormap do?

Before choosing a formula, it is useful to state explicitly what we would like a director-to-color map

$$
\mathbf c:[\mathbf n]\mapsto (R,G,B)
$$

to achieve.

Here $[\mathbf n]$ denotes a nematic orientation: the equivalence class containing both $\mathbf n$ and $-\mathbf n$.

An ideal map would satisfy the following properties.

- **Nematic consistency.**  
  Opposite vectors describe the same physical state, so they must receive exactly the same color:

  $$
  \boxed{\mathbf c(\mathbf n)=\mathbf c(-\mathbf n).}
  $$

  A map that violates this would assign two colors to one physical orientation and would therefore encode the arbitrary sign chosen for the director rather than the nematic state itself.

- **Uniqueness.**  
  Once the nematic identification $\mathbf n\sim-\mathbf n$ has been taken into account, two physically different orientations should receive different colors:

  $$
  [\mathbf n_1]\neq[\mathbf n_2]
  \quad\Longrightarrow\quad
  \mathbf c([\mathbf n_1])\neq\mathbf c([\mathbf n_2]).
  $$

  In practical terms, uniqueness means that a displayed color could, in principle, be decoded back to one and only one nematic orientation.

- **Continuity.**  
  A small change in orientation should produce a small change in color. If

  $$
  [\mathbf n_2]\to[\mathbf n_1],
  $$

  then we want

  $$
  \mathbf c([\mathbf n_2])\to \mathbf c([\mathbf n_1]).
  $$

  Without continuity, a smooth director field could contain artificial color jumps that look like physical discontinuities.

- **Local distinguishability.**  
  Continuity alone is not enough. A nematic orientation has two independent local degrees of freedom. We therefore want both local directions of change in orientation space to remain visible in color space. A small two-dimensional patch of orientations should not collapse into a one-dimensional curve or a single color.

  Differentially, if $D\mathbf c$ is restricted to the two-dimensional tangent plane of the orientation space, we want

  $$
  \operatorname{rank}(D_T\mathbf c)=2.
  $$

- **Perceptual usefulness.**  
  Equal Euclidean distances in encoded RGB do not correspond to equal perceived color differences. A scientifically useful map should therefore be judged in a perceptual color space, not only by algebraic separation in RGB coordinates.

- **Semantic interpretability.**  
  It is useful if the Cartesian axes retain familiar colors:

  $$
  \mathbf e_x\to\text{red},\qquad
  \mathbf e_y\to\text{green},\qquad
  \mathbf e_z\to\text{blue}.
  $$

  These associations make orientation plots easier to read without repeatedly consulting a legend.

- **Displayability.**  
  Every final color must be representable by a normal display:

  $$
  0\le R,G,B\le 1.
  $$

  This should be enforced during the design, rather than obtained by clipping an invalid solution afterward.

## 2. The orientation space is $\mathbb{RP}^2$, not the sphere

A unit vector in three dimensions lies on the sphere

$$
\mathbb S^2=\{\mathbf n\in\mathbb R^3:|\mathbf n|=1\}.
$$

For a polar vector, every point on this sphere represents a different state. A nematic director is different because

$$
\mathbf n\sim-\mathbf n.
$$

Every pair of antipodal points on $\mathbb S^2$ therefore represents one physical orientation. The resulting quotient space is the real projective plane:

$$
\boxed{
\mathbb S^2/\{\mathbf n\sim-\mathbf n\}
\cong
\mathbb{RP}^2.
}
$$

This is the correct domain of any nematic director colormap.

A common temptation is to choose one representative of each antipodal pair, for example by forcing $n_z\ge0$, and then color that hemisphere. This creates a seam. Orientations that are arbitrarily close in $\mathbb{RP}^2$ can lie on opposite sides of the chosen sign convention and suddenly receive unrelated colors. The sign choice therefore trades nematic consistency for an artificial discontinuity rather than solving the problem.

## 3. Why a globally perfect three-dimensional color map is impossible

A displayed color has three coordinates. It is therefore natural to hope that the two-dimensional space $\mathbb{RP}^2$ could simply be placed inside three-dimensional color space without overlap.

Mathematically, nematic consistency, uniqueness, and continuity together ask for an **embedding**

$$
\mathbf c:\mathbb{RP}^2\hookrightarrow\mathbb R^3.
$$

An embedding is a map that is simultaneously continuous, one-to-one, and locally non-degenerate, so that the original space appears as a faithful surface inside the target space.

But the real projective plane cannot be embedded in three-dimensional Euclidean space:

$$
\boxed{\mathbb{RP}^2\not\hookrightarrow\mathbb R^3.}
$$

This is a topological obstruction. It is not a failure of a particular RGB formula, optimizer, or parametrization.

Restricting the target from all of $\mathbb R^3$ to the sRGB cube cannot help, because the sRGB cube is only a subset of $\mathbb R^3$. If an embedding into the larger space does not exist, one cannot obtain one by using a smaller target.

Therefore at least one desired property must be relaxed.

For scientific visualization, Nematics3D keeps:

- nematic consistency,
- continuity,
- local distinguishability,
- perceptual usefulness,
- semantic axis colors,
- and valid sRGB output,

while giving up **global uniqueness**.

That choice leads naturally from an embedding to an immersion.

## 4. Immersion: allow global overlap, preserve local information

An **immersion**

$$
\mathbf c:\mathbb{RP}^2\looparrowright\mathbb R^3
$$

is allowed to intersect itself globally. Two distant orientations may therefore receive the same color. However, the differential remains rank two everywhere:

$$
\boxed{\operatorname{rank}(D_T\mathbf c)=2.}
$$

This distinction is central.

- An **embedding** forbids both local collapse and global self-intersection.
- An **immersion** forbids local collapse but allows global self-intersection.

For a colormap, this means that a globally unique inverse is impossible, but nearby orientations can still be represented by locally distinct colors.

Boy's surface is a classical immersion of $\mathbb{RP}^2$ into $\mathbb R^3$. It therefore provides exactly the kind of topological object required here.

This is also the reason for the function name `n_color_immerse`: the map is intentionally designed as a color-space immersion of nematic orientation space.

## 5. The Boy-type polynomial used by Nematics3D

Nematics3D begins with a fixed polynomial Boy-type immersion

$$
\mathbf p_B(\mathbf n)
=
\bigl(p_1(\mathbf n),p_2(\mathbf n),p_3(\mathbf n)\bigr).
$$

The components are

$$
\begin{aligned}
p_1
&=
\frac12\Big[
(2x^2-y^2-z^2)
+2yz(y^2-z^2)
+zx(x^2-z^2)
+xy(y^2-x^2)
\Big],\\[4pt]
p_2
&=
\frac78\Big[
(y^2-z^2)
+zx(z^2-x^2)
+xy(y^2-x^2)
\Big],\\[4pt]
p_3
&=
\frac18(x+y+z)
\Big[
(x+y+z)^3+4(y-x)(z-y)(x-z)
\Big].
\end{aligned}
$$

Every term is even under the simultaneous sign reversal

$$
(x,y,z)\mapsto(-x,-y,-z),
$$

so

$$
\mathbf p_B(\mathbf n)=\mathbf p_B(-\mathbf n),
$$

as required for a nematic quantity.

The production color map then applies an affine transformation:

$$
\boxed{
\mathbf c_{\rm sRGB}(\mathbf n)
=
A\,\mathbf p_B(\mathbf n)+\mathbf b.
}
$$

This separation is useful. The Boy-type map supplies the required topology; the affine transformation determines how that immersed surface is placed inside color space.

If $A$ is nonsingular, an affine transformation preserves the local immersion property. We can therefore optimize the perceptual placement of an already-valid immersion without having to rediscover a rank-two map from scratch.

## 6. Why optimize in OKLab rather than RGB?

The final output must be sRGB because that is what plotting libraries and displays use. But encoded RGB coordinates are not perceptually uniform.

For example, two RGB pairs with the same Euclidean separation

$$
\|\Delta(R,G,B)\|_2
$$

can appear very different to the human eye depending on where they lie in the color cube.

For optimization and evaluation, the displayed sRGB color is therefore converted to OKLab:

$$
(R,G,B)
\longrightarrow
(L,a,b).
$$

The variables have a useful interpretation:

- $L$ measures perceptual lightness.
- $a$ and $b$ describe the two opponent-color directions.
- The chroma

  $$
  \boxed{C=\sqrt{a^2+b^2}}
  $$

  measures distance from the neutral gray axis.

The mapping is still returned to the user as sRGB. OKLab is the geometry in which perceptual quality is evaluated.

## 7. Measuring local metric fidelity

Local distinguishability only asks that the differential have rank two. That is a binary condition. For visualization we want more: different local orientation directions should ideally be represented with comparable sensitivity.

Let

$$
f(\mathbf n)
=
\operatorname{OKLab}\!\left[
\mathbf c_{\rm sRGB}(\mathbf n)
\right].
$$

At each orientation, restrict the derivative $Df$ to the tangent plane of $\mathbb S^2$. In an orthonormal tangent basis, call this restricted Jacobian $D_Tf$. The color-space metric induced on orientation space is

$$
G=(D_Tf)^T(D_Tf).
$$

If the two eigenvalues of $G$ are equal, infinitesimal orientation changes in all tangent directions are represented equally strongly. If one eigenvalue is much smaller than the other, some orientation changes are visually compressed.

The optimization uses the dimensionless measure

$$
\boxed{
J_{\rm loc}
=
\frac{
\left\langle\operatorname{tr}(G^2)\right\rangle
}{
\left\langle\operatorname{tr}G\right\rangle^2
}
-\frac12.
}
$$

Smaller $J_{\rm loc}$ is better.

The normalization by $\langle\operatorname{tr}G\rangle^2$ is important. Without it, one could make an unnormalized distortion measure appear better simply by shrinking the whole color surface and making all colors nearly identical. The normalized quantity evaluates the *shape* of the local metric rather than rewarding a trivial global rescaling.

## 8. Preserving red, green, and blue axis semantics

A mathematically smooth map can still be inconvenient to read if familiar orientations are assigned arbitrary colors.

Nematics3D therefore treats the Cartesian directions as semantic anchors:

$$
\mathbf e_x\rightarrow \text{red},
\qquad
\mathbf e_y\rightarrow \text{green},
\qquad
\mathbf e_z\rightarrow \text{blue}.
$$

The corresponding target colors are the exact sRGB primaries, but the error is measured after conversion to OKLab.

For an axis $\alpha\in\{x,y,z\}$,

$$
d_\alpha
=
\left\|
\operatorname{OKLab}[\mathbf c(\mathbf e_\alpha)]
-
\operatorname{OKLab}[\mathrm{target}_\alpha]
\right\|_2.
$$

The current optimization does not require the axes to equal the primaries exactly. Instead it constrains each perceptual deviation to remain below a calibrated tolerance. This allows modest movement of the three anchors when that movement substantially improves the rest of the colormap.

## 9. The first optimization and the $J_{\rm loc}=0.43$ solution

The first completed OKLab optimization considered two competing goals:

1. preserve local metric fidelity;
2. keep the Cartesian axis colors close to red, green, and blue.

It minimized the total axis-color error

$$
J_{\rm axis}
=
\sum_{\alpha=x,y,z}
\left\|
\operatorname{OKLab}[\mathbf c(\mathbf e_\alpha)]
-
\operatorname{OKLab}[\mathbf e_\alpha]
\right\|^2
$$

subject to

$$
J_{\rm loc}\le t
$$

and the sRGB gamut constraint.

Scanning $t$ produced a Pareto frontier. After normalizing the two objectives over the scanned range, the geometric knee occurred near

$$
\boxed{J_{\rm loc}=0.43.}
$$

That solution was a defensible answer to the **two-objective** question

> How should we trade local metric fidelity against axis-color fidelity?

However, it was not yet a satisfactory answer to the broader visualization problem.

## 10. Why the $0.43$ solution was not vivid enough

Visual inspection exposed a systematic weakness of the $0.43$ map: too much of orientation space lay close to the neutral axis in OKLab. Large regions therefore appeared gray, muted, or muddy.

This does not mean the original optimizer failed. It means the objective function was incomplete.

A Pareto optimization can only optimize quantities that appear in its objectives or constraints. Neither $J_{\rm loc}$ nor $J_{\rm axis}$ explicitly rewards chromatic colors away from the neutral axis.

The revised design therefore adds **vividness** as an explicit objective. The relevant quantity is the OKLab chroma

$$
C=\sqrt{a^2+b^2}.
$$

Useful diagnostics include

$$
\langle C\rangle,
$$

the median and quantiles of $C$, and low-chroma fractions such as

$$
P(C<0.10).
$$

The design question becomes:

> How much chroma can be gained while preserving controlled local distortion, recognizable axis colors, and valid sRGB output?

## 11. Calibrating an acceptable axis-color error

A tolerance is more meaningful when it is tied to an existing visual reference rather than chosen as an arbitrary number.

The earlier production version of `n_color_immerse()` mapped the $x$ axis to approximately

$$
(0.90536,\;0.22875,\;0.22063).
$$

Its OKLab distance from exact sRGB red is approximately

$$
\boxed{\delta_{\rm axis}=0.051845.}
$$

That previous map had already demonstrated that an axis deviation of roughly this magnitude remained visually recognizable as the intended semantic color.

The revised optimization therefore imposes

$$
d_{\rm OKLab}
\bigl(
\mathbf c(\mathbf e_\alpha),
\mathrm{target}_\alpha
\bigr)
\le
\delta_{\rm axis},
\qquad
\alpha=x,y,z.
$$

The old map is used here only to calibrate a tolerable semantic error. It is not assumed to be geometrically or perceptually optimal.

## 12. Revised optimization: maximize vividness under controlled distortion

Within the affine family

$$
\mathbf c_{\rm sRGB}(\mathbf n)
=
A\mathbf p_B(\mathbf n)+\mathbf b,
$$

the revised optimization problem is

$$
\boxed{
\begin{aligned}
\max_{A,\mathbf b}\quad
&\langle C(\mathbf n)\rangle\\
\text{subject to}\quad
&J_{\rm loc}\le t,\\
&d_{\rm OKLab}(\mathbf c(\mathbf e_x),\mathrm{red})
\le\delta_{\rm axis},\\
&d_{\rm OKLab}(\mathbf c(\mathbf e_y),\mathrm{green})
\le\delta_{\rm axis},\\
&d_{\rm OKLab}(\mathbf c(\mathbf e_z),\mathrm{blue})
\le\delta_{\rm axis},\\
&\mathbf c_{\rm sRGB}(\mathbf n)\in[0,1]^3
\quad
\forall[\mathbf n]\in\mathbb{RP}^2.
\end{aligned}
}
$$

Here $t$ controls how much local geometric distortion is accepted in exchange for greater vividness.

This point is easy to miss: the old $0.43$ solution and the current $0.55$ solution do **not** come from the same objective function. The number $0.55$ should therefore not be interpreted as simply choosing a worse point on the old Pareto curve. It belongs to a revised optimization in which chroma is the quantity being maximized.

## 13. Why the production map uses $J_{\rm loc}\le0.55$

Increasing the allowed local distortion gives the optimizer more freedom to move the immersed surface toward chromatic regions of the sRGB gamut.

Three representative regimes are useful conceptually:

- Around $J_{\rm loc}=0.43$, the map prioritizes local metric regularity strongly, but retains too much low-chroma area.
- Around $J_{\rm loc}=0.70$, considerably more distortion is permitted. The map can become more vivid, but local geometric fidelity is sacrificed more than necessary.
- The selected $J_{\rm loc}=0.55$ solution provides the preferred compromise: it produces a clear improvement in vividness over the pre-chroma baseline while retaining controlled local distortion and all three semantic axis constraints.

For that reason the current Nematics3D production map is the vividness-optimized solution satisfying

$$
\boxed{J_{\rm loc}\le0.55.}
$$

The value $0.55$ is therefore a design choice on a perceptual-geometric trade-off, not a universal topological constant.

## 14. Why gamut validity is a hard constraint

An optimizer working freely in a perceptual color space can easily propose colors that no sRGB display can represent.

A tempting workaround is to optimize first and then clip:

$$
R\mapsto\min(1,\max(0,R)),
$$

and similarly for $G$ and $B$.

That is undesirable here. Clipping is nonlinear and can flatten a finite region of the optimized surface onto a face, edge, or corner of the RGB cube. A map that had good local geometry before clipping may therefore acquire compressed or locally collapsed regions afterward.

Nematics3D instead treats gamut membership as part of the optimization:

$$
\boxed{
0\le
c_{{\rm sRGB},i}(\mathbf n)
\le1
\quad
\text{for every orientation and every channel }i.
}
$$

The optimization uses sampled orientation sets and dense verification. Candidate solutions are checked over a much denser set of directions; any gamut-violating directions can be added back to the active constraint set and the optimization repeated.

The final implementation still applies `np.clip(..., 0, 1)` as a numerical safety guard, but the selected map is designed to lie in gamut before that guard is applied.

## 15. The production affine transformation

The current implementation evaluates the Boy-type polynomial and then applies

$$
\mathbf c_{\rm sRGB}
=
A\mathbf p_B+\mathbf b,
$$

with

$$
A=
\begin{pmatrix}
0.5022508927 & 0.0814191820 & 0.4278817283\\
-0.2622468169 & 0.4198664553 & 0.2843783906\\
-0.2603273419 & -0.3829942093 & 0.3705024139
\end{pmatrix}
$$

and

$$
\mathbf b=
\begin{pmatrix}
0.3810134663\\
0.4051244319\\
0.4114207202
\end{pmatrix}.
$$

These coefficients are not intended to carry an independent physical interpretation. They encode the selected perceptual placement of the Boy-type immersion inside the sRGB cube.

The essential structure is therefore:

$$
\boxed{
\text{nematic director}
\longrightarrow
\text{Boy-type immersion}
\longrightarrow
\text{optimized affine placement}
\longrightarrow
\text{sRGB color}.
}
$$

## 16. How to interpret the resulting colors

The map should be read as a **local orientation encoding**, not as a globally invertible coordinate system.

A few practical rules are important:

- $\mathbf n$ and $-\mathbf n$ always have the same color.
- Nearby colors usually provide useful information about nearby orientations because local rank is preserved and local distortion is explicitly controlled.
- Red-like, green-like, and blue-like colors are anchored to the $x$, $y$, and $z$ directions.
- Two well-separated orientations can, in principle, share the same color because any immersion of $\mathbb{RP}^2$ into three dimensions must self-intersect somewhere.
- Therefore a color alone should not be treated as a globally unique label for a director.
- The map is intended primarily for reading spatial structure, gradients, domains, distortions, and defects in a director field.

This is the compromise enforced by topology: global ambiguity is accepted so that local structure remains continuous and informative.

## 17. Direct usage

In [ ]:
import numpy as np
from nematics3d.field import n_color_immerse

directors = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
])

colors = n_color_immerse(directors)
colors

Because the mapping is nematic, reversing every director must leave the colors unchanged:

In [ ]:
np.allclose(
    np.asarray(n_color_immerse(directors)),
    np.asarray(n_color_immerse(-directors)),
)

## 18. Visualizing the full colormap

A useful way to inspect the mapping is to color a sphere of director representatives with `n_color_immerse()`.

Remember that antipodal points correspond to the same nematic orientation and therefore must have identical colors. The resulting sphere is consequently a doubled representation of the physical space $\mathbb{RP}^2$, but it is often easier to visualize than the projective plane directly.

When inspecting such a color sphere, look for:

- antipodal color equality;
- smooth variation without artificial seams;
- recognizable red, green, and blue neighborhoods around the Cartesian axes;
- avoidance of large gray or muddy regions;
- and the unavoidable global color coincidences associated with self-intersection of the immersion.

For the optimization history and reproducibility scripts behind the selected affine transform, see the `rp2_colormap` project and its `doc/manuscript/colormap_selection/why_this_colormap.ipynb` notebook.